Day 3 目标：加入 LLM 分类节点，但保留规则分类兜底

今天的核心目标是：

把 Day 2 的规则分类 classify_task 升级成：
优先用 LLM 判断任务类型；如果 LLM 不可用 / 输出异常，就回退到规则分类。

也就是说，今天不是做 RAG，不是做复杂 Agent，只是让你的：

用户输入 → classify_task → route_task → 对应节点

里的 classify_task 更智能一点。

# 一、Day 3 你要完成什么？

今天完成后，你的 Agent 应该支持：

用户输入：
帮我根据实验结果写一段组会汇报

LLM 分类：
report_generation

路由：
report_node

输出：
这是汇报生成任务，后续会接入 Report Writer。

并且如果 LLM 调用失败，也不会崩：

LLM 失败
↓
自动使用规则分类
↓
继续正常运行

这个兜底机制非常重要。Agent 项目里，能优雅失败比“看起来高级”更值钱。

# 二、Day 3 最终目录结构

在 Day 2 基础上，今天建议新增两个文件：

F:\ResearchAgent
├── .env
├── run_cli.py
└── src
    └── research_agent
        └── graph
            ├── state.py
            ├── nodes.py
            ├── router.py
            ├── workflow.py
            └── llm_classifier.py

今天重点文件：

| 文件                  | 今天做什么                                     |
| ------------------- | ----------------------------------------- |
| `.env`              | 保存 API 配置                                 |
| `llm_classifier.py` | 新增 LLM 分类逻辑                               |
| `nodes.py`          | 修改 `classify_task`，优先调用 LLM               |
| `state.py`          | 可选新增 `route_reason` / `classifier_source` |
| `run_cli.py`        | 基本不用大改                                    |


# 三、先安装依赖

在你的项目环境里执行：

cd F:\ResearchAgent
.\.conda\python.exe -m pip install langchain-openai python-dotenv

如果你的 .conda 已经能正常激活，也可以用：

conda activate .\.conda
python -m pip install langchain-openai python-dotenv

# 四、创建 .env

在项目根目录创建：

F:\ResearchAgent\.env

内容先写成这样：

OPENAI_API_KEY=你的_api_key
OPENAI_BASE_URL=你的_base_url
OPENAI_MODEL=gpt-4o-mini
USE_LLM_CLASSIFIER=true

如果你暂时不想接 API，也可以先写：

USE_LLM_CLASSIFIER=false

这样今天代码仍然能跑，会自动使用规则分类。

# 五、修改 state.py

路径：

F:\ResearchAgent\src\research_agent\graph\state.py

建议改成：

from typing import TypedDict


class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str

    # Day 3 新增：记录分类来源，方便调试
    classifier_source: str
    route_reason: str

这两个字段不是必须，但我建议加上：

字段	作用
classifier_source	记录是 llm 还是 rule
route_reason	记录为什么分到这个任务类型

这样你后面调试 Agent 会舒服很多。

# 六、新建 llm_classifier.py

路径：

F:\ResearchAgent\src\research_agent\graph\llm_classifier.py

复制下面代码：

import json
import os
from typing import Dict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


VALID_TASK_TYPES = {
    "paper_question",
    "experiment_analysis",
    "dataset_recommendation",
    "report_generation",
    "code_help",
    "general",
}


def get_llm_classifier_enabled() -> bool:
    """
    是否启用 LLM 分类器。
    通过 .env 里的 USE_LLM_CLASSIFIER 控制。
    """
    load_dotenv()
    value = os.getenv("USE_LLM_CLASSIFIER", "false").lower()
    return value in {"1", "true", "yes", "y"}


def build_classifier_prompt(query: str) -> str:
    """
    构造任务分类 Prompt。
    要求模型只输出 JSON，方便解析。
    """
    return f"""
你是一个科研 Agent 的任务分类器。

请根据用户输入，将任务分类为以下六类之一：

1. paper_question
   - 论文总结、论文问答、文献解释、论文对比

2. experiment_analysis
   - 实验结果分析、CSV/JSONL 输出分析、benchmark 结果解释、幻觉指标分析

3. dataset_recommendation
   - 数据集推荐、数据集选择、数据集构建建议

4. report_generation
   - 组会汇报、PPT 文案、阶段总结、实验汇报文本

5. code_help
   - 代码解释、代码报错、脚本修改、环境问题、bug 修复

6. general
   - 其他通用科研助手问题

用户输入：
{query}

请只输出 JSON，不要输出 Markdown，不要解释。

JSON 格式如下：
{{
  "task_type": "paper_question | experiment_analysis | dataset_recommendation | report_generation | code_help | general",
  "reason": "一句话说明分类原因"
}}
""".strip()


def classify_with_llm(query: str) -> Dict[str, str]:
    """
    使用 LLM 对用户问题进行分类。
    如果调用失败或解析失败，会抛出异常，由上层 fallback。
    """
    load_dotenv()

    model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    base_url = os.getenv("OPENAI_BASE_URL") or None

    llm = ChatOpenAI(
        model=model_name,
        base_url=base_url,
        temperature=0,
    )

    prompt = build_classifier_prompt(query)
    response = llm.invoke(prompt)

    content = response.content.strip()

    data = json.loads(content)

    task_type = data.get("task_type", "").strip()
    reason = data.get("reason", "").strip()

    if task_type not in VALID_TASK_TYPES:
        raise ValueError(f"Invalid task_type from LLM: {task_type}")

    return {
        "task_type": task_type,
        "reason": reason,
    }

# 七、修改 nodes.py

路径：

F:\ResearchAgent\src\research_agent\graph\nodes.py

今天建议把规则分类单独拆出来：

from .state import AgentState
from .llm_classifier import classify_with_llm, get_llm_classifier_enabled


def classify_task_by_rule(query: str) -> dict:
    """
    规则分类器：作为 LLM 分类失败后的兜底。
    """
    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
        reason = "命中了论文 / paper 关键词。"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query or "benchmark" in query.lower():
        task_type = "experiment_analysis"
        reason = "命中了实验、COCO、幻觉或 benchmark 相关关键词。"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
        reason = "命中了数据集 / dataset 关键词。"
    elif "汇报" in query or "PPT" in query.upper() or "组会" in query or "总结" in query:
        task_type = "report_generation"
        reason = "命中了汇报、PPT、组会或总结相关关键词。"
    elif "代码" in query or "脚本" in query or "bug" in query.lower() or "报错" in query or "环境" in query:
        task_type = "code_help"
        reason = "命中了代码、脚本、bug、报错或环境相关关键词。"
    else:
        task_type = "general"
        reason = "未命中明确任务关键词，归为通用科研助手任务。"

    return {
        "task_type": task_type,
        "reason": reason,
    }


def classify_task(state: AgentState) -> dict:
    """
    Day 3 分类节点：
    优先使用 LLM 分类；
    如果关闭 LLM 或 LLM 失败，则回退到规则分类。
    """
    query = state["query"]

    if get_llm_classifier_enabled():
        try:
            llm_result = classify_with_llm(query)
            return {
                "task_type": llm_result["task_type"],
                "classifier_source": "llm",
                "route_reason": llm_result["reason"],
            }
        except Exception as e:
            rule_result = classify_task_by_rule(query)
            return {
                "task_type": rule_result["task_type"],
                "classifier_source": "rule_fallback",
                "route_reason": f"LLM 分类失败，回退规则分类。错误：{str(e)}；规则原因：{rule_result['reason']}",
            }

    rule_result = classify_task_by_rule(query)
    return {
        "task_type": rule_result["task_type"],
        "classifier_source": "rule",
        "route_reason": rule_result["reason"],
    }

然后保留你 Day 2 已经写好的节点：

def paper_node(state: AgentState) -> dict:
    return {
        "result": "这是论文问答任务，后续会接入论文 RAG。"
    }


def experiment_node(state: AgentState) -> dict:
    return {
        "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
    }


def dataset_node(state: AgentState) -> dict:
    return {
        "result": "这是数据集推荐任务，后续会接入数据集资料库。"
    }


def report_node(state: AgentState) -> dict:
    return {
        "result": "这是汇报生成任务，后续会接入 Report Writer。"
    }


def code_node(state: AgentState) -> dict:
    return {
        "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
    }


def general_node(state: AgentState) -> dict:
    return {
        "result": "这是通用科研助手任务。"
    }


def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
分类来源：{state["classifier_source"]}
分类原因：{state["route_reason"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }

# 八、检查 router.py

如果你 Day 2 已经加了 report_generation，这里应该已经是这样：

from typing import Literal

from .state import AgentState


def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "report_node",
    "code_node",
    "general_node",
]:
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "report_generation":
        return "report_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"

# 九、检查 workflow.py

确保你已经注册了 report_node：

from langgraph.graph import StateGraph, START, END

from .state import AgentState
from .nodes import (
    classify_task,
    paper_node,
    experiment_node,
    dataset_node,
    report_node,
    code_node,
    general_node,
    final_answer_node,
)
from .router import route_task


def build_graph():
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_task", classify_task)

    workflow.add_node("paper_node", paper_node)
    workflow.add_node("experiment_node", experiment_node)
    workflow.add_node("dataset_node", dataset_node)
    workflow.add_node("report_node", report_node)
    workflow.add_node("code_node", code_node)
    workflow.add_node("general_node", general_node)

    workflow.add_node("final_answer", final_answer_node)

    workflow.add_edge("__start__", "classify_task")

    workflow.add_conditional_edges(
        "classify_task",
        route_task,
        {
            "paper_node": "paper_node",
            "experiment_node": "experiment_node",
            "dataset_node": "dataset_node",
            "report_node": "report_node",
            "code_node": "code_node",
            "general_node": "general_node",
        },
    )

    workflow.add_edge("paper_node", "final_answer")
    workflow.add_edge("experiment_node", "final_answer")
    workflow.add_edge("dataset_node", "final_answer")
    workflow.add_edge("report_node", "final_answer")
    workflow.add_edge("code_node", "final_answer")
    workflow.add_edge("general_node", "final_answer")

    workflow.add_edge("final_answer", "__end__")

    return workflow.compile()

你也可以继续用：

workflow.add_edge(START, "classify_task")
workflow.add_edge("final_answer", END)

这更推荐，因为你已经导入了：

from langgraph.graph import StateGraph, START, END

所以建议写成：

workflow.add_edge(START, "classify_task")
workflow.add_edge("final_answer", END)

# 十、修改 run_cli.py

因为 state.py 新增了两个字段，所以初始 State 也要补上：

from src.research_agent.graph.workflow import build_graph


def run_agent(query: str) -> dict:
    graph = build_graph()

    result = graph.invoke({
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
        "classifier_source": "",
        "route_reason": "",
    })

    return result


def main():
    print("ResearchAgent v0.1 CLI")
    print("输入 q / quit / exit 退出")

    while True:
        query = input("\n请输入你的科研问题：\n> ")

        if query.lower() in ["q", "quit", "exit"]:
            print("已退出。")
            break

        result = run_agent(query)

        print("\n===== Agent 输出 =====")
        print(result["final_answer"])


if __name__ == "__main__":
    main()

# 十一、先用规则分类测试

先把 .env 里设置成：

USE_LLM_CLASSIFIER=false

然后运行：

cd F:\ResearchAgent
.\.conda\python.exe run_cli.py

测试：

请帮我分析 coco_val_n300_g1 的幻觉风险

预期看到：

任务类型：experiment_analysis
分类来源：rule
分类原因：命中了实验、COCO、幻觉或 benchmark 相关关键词。
处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。

再测试：

帮我生成组会汇报文本

预期：

任务类型：report_generation
分类来源：rule
分类原因：命中了汇报、PPT、组会或总结相关关键词。
处理结果：这是汇报生成任务，后续会接入 Report Writer。

# 十二、再打开 LLM 分类测试

确认 .env 有 API 配置后，把它改成：

USE_LLM_CLASSIFIER=true

然后运行：

.\.conda\python.exe run_cli.py

测试一些规则不容易命中的输入：

我想把最近的实验进展整理成一段给导师看的说明

理想情况下，LLM 会分类为：

report_generation

因为它理解“整理实验进展给导师看”更像汇报生成。

再测试：

这个错误 ModuleNotFoundError: No module named langgraph 怎么解决

理想分类：

code_help

再测试：

我想找一些适合验证视觉语言模型幻觉的数据来源

理想分类：

dataset_recommendation

# 十三、Day 3 常见错误
错误 1：ModuleNotFoundError: No module named 'langchain_openai'

执行：

.\.conda\python.exe -m pip install langchain-openai
错误 2：.env 没读到

确认 .env 在：

F:\ResearchAgent\.env

不是在：

F:\ResearchAgent\src\.env
错误 3：KeyError: 'classifier_source'

说明你改了 state.py，但是 run_cli.py 的初始 State 没加：

"classifier_source": "",
"route_reason": "",

补上即可。

错误 4：LLM 输出不是 JSON

暂时不用怕，因为我们已经有 fallback：

LLM 分类失败，回退规则分类

这个本身就是今天要实现的能力之一。

# 十五、今天你真正要理解的东西

Day 3 的重点不是 API，而是这个设计思想：

智能组件可以失败，所以必须有兜底。

今天的分类节点应该形成这个结构：

classify_task
├── 如果 USE_LLM_CLASSIFIER=true
│   ├── 尝试 LLM 分类
│   ├── 成功：使用 LLM 结果
│   └── 失败：回退规则分类
└── 如果 USE_LLM_CLASSIFIER=false
    └── 使用规则分类

一句话记住：

LLM 负责增强理解能力，规则负责保证系统稳定运行。

这就是一个像样 Agent 项目的味道了。